In [ ]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
os.listdir("/kaggle/input/garbage-classification/garbage_classification/")

In [ ]:
import os
import matplotlib.pyplot as plt

DATASET_PATH = "/kaggle/input/garbage-classification/garbage_classification"

class_counts = {}

for class_name in sorted(os.listdir(DATASET_PATH)):
    class_path = os.path.join(DATASET_PATH, class_name)

    if os.path.isdir(class_path):
        images = [
            f for f in os.listdir(class_path)
            if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
        ]
        class_counts[class_name] = len(images)

print("Total Classes:", len(class_counts))
print("Total Images:", sum(class_counts.values()))

for name, count in class_counts.items():
    print(f"{name}: {count}")

plt.figure(figsize=(12, 5))
plt.bar(class_counts.keys(), class_counts.values())
plt.xticks(rotation=45, ha="right")
plt.ylabel("Number of Images")
plt.title("Images per Waste Category")
plt.tight_layout()
plt.show()

In [ ]:

import os
import random
import matplotlib.pyplot as plt
from PIL import Image

DATASET_PATH = "/kaggle/input/garbage-classification/garbage_classification"

random.seed(42)
classes = sorted(os.listdir(DATASET_PATH))

plt.figure(figsize=(15, 10))

for i, class_name in enumerate(classes):
    class_path = os.path.join(DATASET_PATH, class_name)
    images = [
        f for f in os.listdir(class_path)
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))
    ]

    if images:
        image_path = os.path.join(class_path, random.choice(images))
        img = Image.open(image_path).convert("RGB")

        plt.subplot(3, 4, i + 1)
        plt.imshow(img)
        plt.title(class_name)
        plt.axis("off")

plt.tight_layout()
plt.show()


In [ ]:

%%time

import os
import pandas as pd
import numpy as np

DATASET_PATH = "/kaggle/input/garbage-classification/garbage_classification"

records = []

for class_name in sorted(os.listdir(DATASET_PATH)):
    class_path = os.path.join(DATASET_PATH, class_name)

    if not os.path.isdir(class_path):
        continue

    for photo in os.listdir(class_path):
        if photo.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
            records.append({
                "path": os.path.join(class_path, photo),
                "target": class_name
            })

data = pd.DataFrame(records)

print("Total images:", len(data))
print("Total classes:", data["target"].nunique())
print(data.head())


In [ ]:
data.shape

In [ ]:
data.duplicated().sum()

In [ ]:
data = data.sample(frac=1)
data.reset_index(drop=True, inplace=True)
data.head()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.countplot(data['target'])
plt.xticks(rotation="45")

In [ ]:
data['target'].value_counts()

In [ ]:
import matplotlib.image as mpimg

for i in range(100):
    image = mpimg.imread(data.loc[i, 'path'])
    plt.imshow(image)
    plt.show()

In [ ]:
total_counts = 0

for class_ in os.listdir("/kaggle/input/garbage-classification/garbage_classification/"):
    count_class = 0
    for photo in os.listdir("/kaggle/input/garbage-classification/garbage_classification/" + class_):
        count_class += 1
        total_counts += 1
    print(str(class_) + " has " + str(count_class) + " photos")

In [ ]:
total_counts

In [ ]:
weights_ = dict()
for class_ in os.listdir("/kaggle/input/garbage-classification/garbage_classification/"):
    count_class = 0
    for photo in os.listdir("/kaggle/input/garbage-classification/garbage_classification/" + class_):
        count_class += 1
    print(str(class_) + " has " + str((count_class/total_counts) * 100) + "% of photos")
    weights_[class_] = 1 - (count_class/total_counts)

In [ ]:
weights_

In [ ]:

from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data["target"]
)

print("Training images:", len(X_train))
print("Testing images:", len(X_test))
print("Training classes:", X_train["target"].nunique())
print("Testing classes:", X_test["target"].nunique())


In [ ]:

from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    X_train,
    test_size=0.125,
    random_state=42,
    stratify=X_train["target"]
)

print("Training:", len(train_df))
print("Validation:", len(val_df))
print("Testing:", len(X_test))


In [ ]:

import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import (
    MobileNetV2, preprocess_input
)
from tensorflow.keras import layers, models
from tensorflow.keras.utils import Sequence
from PIL import Image
import numpy as np

IMG_SIZE = 224
BATCH_SIZE = 32
NUM_CLASSES = data["target"].nunique()

print("TensorFlow:", tf.__version__)
print("Classes:", NUM_CLASSES)
print("GPU:", tf.config.list_physical_devices("GPU"))


In [ ]:

from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

valid_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_gen = train_datagen.flow_from_dataframe(
    train_df,
    x_col="path",
    y_col="target",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True,
    seed=42
)

val_gen = valid_datagen.flow_from_dataframe(
    val_df,
    x_col="path",
    y_col="target",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=list(train_gen.class_indices.keys()),
    shuffle=False
)

test_gen = valid_datagen.flow_from_dataframe(
    X_test,
    x_col="path",
    y_col="target",
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    classes=list(train_gen.class_indices.keys()),
    shuffle=False
)


In [ ]:

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
    layers.Lambda(preprocess_input),
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()


In [ ]:

# callbacks = [
#     tf.keras.callbacks.EarlyStopping(
#         monitor="val_loss",
#         patience=2,
#         restore_best_weights=True
#     ),
#     tf.keras.callbacks.ModelCheckpoint(
#         "best_garbage_model.keras",
#         monitor="val_loss",
#         save_best_only=True
#     )
# ]

# history = model.fit(
#     train_gen,
#     validation_data=val_gen,
#     epochs=8,
#     callbacks=callbacks
# )


In [ ]:
base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(224, 224, 3)),
    base_model,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(12, activation="softmax")
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(X_train["target"])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=X_train["target"]
)

cw = {
    train_gen.class_indices[class_name]: weight
    for class_name, weight in zip(classes, weights)
}

print(cw)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True
    ),

    tf.keras.callbacks.ModelCheckpoint(
        "/kaggle/working/best_garbage_model.keras",
        monitor="val_loss",
        save_best_only=True
    )
]

print("Callbacks ready!")

In [ ]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=5,
    callbacks=callbacks,
    class_weight=cw
)

In [ ]:
test_loss, test_acc = model.evaluate(test_gen)

print("Test Loss:", test_loss)
print("Test Accuracy:", round(test_acc * 100, 2), "%")

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

test_gen.reset()

predictions = model.predict(test_gen)
y_pred = np.argmax(predictions, axis=1)
y_true = test_gen.classes

labels = list(test_gen.class_indices.keys())

print(classification_report(
    y_true,
    y_pred,
    target_names=labels
))

plt.figure(figsize=(12, 10))

sns.heatmap(
    confusion_matrix(y_true, y_pred),
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=labels,
    yticklabels=labels
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Garbage Classification - Confusion Matrix")
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.show()

In [ ]:
model.save("/kaggle/working/garbage_classification_model.h5")

print("Model saved successfully!")

In [ ]:
# Save model architecture separately
model_json = model.to_json()

with open("/kaggle/working/model_architecture.json", "w") as json_file:
    json_file.write(model_json)

# Save trained weights separately
model.save_weights("/kaggle/working/model_weights.weights.h5")

print("Architecture and weights saved successfully!")